In [1]:
#Récupération des artistes avec MusicBrainz
import time
import requests
import pandas as pd

headers = {"User-Agent": "projet-python (ton-email@exemple.fr)"}
artistes = ["Ninho", "Booba", "Jul", "PNL", "SCH"]

lignes = []
for nom in artistes:
    r = requests.get(
        "https://musicbrainz.org/ws/2/artist/",
        params={"query": f'artist:"{nom}"', "fmt": "json", "limit": 1},
        headers=headers,
    )
    a = r.json()["artists"][0]
    lignes.append({
        "nom": a["name"],
        "pays": a.get("country"),
        "ville": a.get("begin-area", {}).get("name") or a.get("area", {}).get("name"),
    })
    time.sleep(1)  # MusicBrainz limite à environ 1 requête par seconde

df = pd.DataFrame(lignes)
print(df)

     nom pays                 ville
0  Ninho   FR            Longjumeau
1  Booba   FR  Boulogne-Billancourt
2    Jul   FR             Marseille
3    PNL   FR      Corbeil-Essonnes
4    SCH   FR             Marseille


In [2]:
#Transformer la ville en coordonnées
from geopy.geocoders import Nominatim

geo = Nominatim(user_agent="projet-python")

def coords(ville):
    if not ville:
        return pd.Series([None, None])
    loc = geo.geocode(f"{ville}, France")
    time.sleep(1)
    return pd.Series([loc.latitude, loc.longitude]) if loc else pd.Series([None, None])

df[["lat", "lon"]] = df["ville"].apply(coords)

In [3]:
#Faire une carte
import folium

carte = folium.Map(location=[46.6, 2.5], zoom_start=6)

for _, ligne in df.dropna(subset=["lat", "lon"]).iterrows():
    folium.CircleMarker(
        location=[ligne["lat"], ligne["lon"]],
        radius=8,
        popup=f'{ligne["nom"]} ({ligne["ville"]})',
        fill=True,
    ).add_to(carte)

carte.save("carte_origines.html")